In [4]:
# delimitar_bacia.py
import os
import tempfile
import json
from pathlib import Path

import rasterio
from rasterio.warp import transform
from shapely.geometry import Point, mapping
import geopandas as gpd
from whitebox.whitebox_tools import WhiteboxTools

In [ ]:
# =========================
# CONFIGURAÇÃO
# =========================
DEM_PATH = r"C:\Users\gabriel.coimbra\Desktop\Análise das Inequações\10.09.25_Eu_SãoSepé\mdt\mdt_recortado.tif"   # GeoTIFF do MDE
# Exutório em WGS84 (lon/lat). Ex.: perto de um rio conhecido.
EXUTORIO_LAT, EXUTORIO_LON = -30.1333931,-53.5752407

# Parâmetros
STREAM_THRESH_ACC = 1000   # limiar de fluxo p/ formar drenagem (em "células" de acumulação)
SNAP_DIST_CELLS = 5        # raio de busca p/ snap do exutório (em células)
SAIDA_VETOR = r"C:\Users\gabriel.coimbra\Desktop\Análise das Inequações\10.09.25_Eu_SãoSepé\mdt\bacia.geojson"   # saída da bacia em GeoJSON
SAIDA_GPKG  = r"C:\Users\gabriel.coimbra\Desktop\Análise das Inequações\10.09.25_Eu_SãoSepé\mdt\bacia.gpkg"      # opcional (GeoPackage)
EXPORTAR_REDE = True       # exportar rede de drenagem (linhas)

# =========================
# PREPARO
# =========================
wbt = WhiteboxTools()
workdir = tempfile.mkdtemp(prefix="wbt_bacia_")
wbt.work_dir = workdir

# Arquivos temporários
dem_filled = os.path.join(workdir, "dem_filled.tif")
fdir = os.path.join(workdir, "d8_pointer.tif")
facc = os.path.join(workdir, "d8_acc.tif")
streams = os.path.join(workdir, "streams.tif")
pour_points = os.path.join(workdir, "pour_points.shp")
pour_points_snapped = os.path.join(workdir, "pour_points_snapped.shp")
watershed_r = os.path.join(workdir, "watershed.tif")
watershed_v = os.path.join(workdir, "watershed.shp")
streams_vec = os.path.join(workdir, "streams.shp")

# =========================
# 1) Ler CRS do DEM e reprojetar exutório
# =========================
with rasterio.open(DEM_PATH) as src:
    dem_crs = src.crs
    dem_transform = src.transform
    # reprojeta lon/lat -> CRS do DEM
    (x_ex), (y_ex) = transform(
        "EPSG:4326",
        dem_crs,
        [EXUTORIO_LON],
        [EXUTORIO_LAT]
    )
    x_ex, y_ex = x_ex[0], y_ex[0]

# Salva exutório como shapefile no CRS do DEM
gdf_pt = gpd.GeoDataFrame(
    {"id": [1]},
    geometry=[Point(x_ex, y_ex)],
    crs=dem_crs
)
gdf_pt.to_file(pour_points)

# =========================
# 2) Hidroprocessamento
# =========================

# 2.1 Breachar/Preencher depressões
# (esta função aceita 'dem=' no wrapper)
wbt.breach_depressions(dem=DEM_PATH, output=dem_filled, max_depth=100.0)

# 2.2 Direção de fluxo (D8) e Acumulação
# d8_pointer aceita 'dem='
wbt.d8_pointer(dem=dem_filled, output=fdir)

# d8_flow_accumulation NÃO aceita 'dem=' em algumas versões do wrapper;
# use 'i=' (input) e passe o pntr pré-calculado.
wbt.d8_flow_accumulation(
    i=dem_filled,
    output=facc,
    out_type="cells",
    pntr=fdir
)

# 2.3 Extrair rede de drenagem (a partir da acumulação)
wbt.extract_streams(
    flow_accum=facc,
    output=streams,
    threshold=STREAM_THRESH_ACC
)

# 2.4 Snap do exutório para a célula de maior drenagem próxima
wbt.snap_pour_points(
    pour_pts=pour_points,
    flow_accum=facc,
    output=pour_points_snapped,
    snap_dist=SNAP_DIST_CELLS
)

# 2.5 Delimitar a bacia (usa a direção de fluxo D8 + ponto "snapped")
wbt.watershed(
    d8_pntr=fdir,
    pour_pts=pour_points_snapped,
    output=watershed_r
)

# 2.6 Converter a bacia de raster para polígono
wbt.raster_to_vector_polygons(
    i=watershed_r,
    output=watershed_v,
    simplification=0.0
)


# =========================
# 3) Exportar resultados (GeoJSON/GPKG)
# =========================
bacia = gpd.read_file(watershed_v).to_crs("EPSG:4326")
bacia.to_file(SAIDA_VETOR, driver="GeoJSON")
if SAIDA_GPKG:
    Path(SAIDA_GPKG).parent.mkdir(parents=True, exist_ok=True)
    bacia.to_file(SAIDA_GPKG, layer="bacia", driver="GPKG")

if EXPORTAR_REDE:
    # Converte a rede raster -> vetorial (linhas)
    wbt.raster_streams_to_vector(streams=streams, d8_pntr=fdir, output=streams_vec)
    rede = gpd.read_file(streams_vec).to_crs("EPSG:4326")
    # salva no mesmo GPKG (se indicado)
    if SAIDA_GPKG:
        rede.to_file(SAIDA_GPKG, layer="drenagem", driver="GPKG")
    # ou salva ao lado do GeoJSON
    rede.to_file(os.path.splitext(SAIDA_VETOR)[0] + "_rede.geojson", driver="GeoJSON")

print("Concluído.")
print(f"Bacia em: {SAIDA_VETOR}")
if SAIDA_GPKG:
    print(f"Também em: {SAIDA_GPKG}")


.\whitebox_tools.exe --run="BreachDepressions" --wd="C:\Users\GABRIE~1.COI\AppData\Local\Temp\wbt_bacia_fpb9vppk" --dem='C:\Users\gabriel.coimbra\Desktop\Análise das Inequações\10.09.25_Eu_SãoSepé\mdt\mdt_recortado.tif' --output='C:\Users\GABRIE~1.COI\AppData\Local\Temp\wbt_bacia_fpb9vppk\dem_filled.tif' --max_depth='100.0' -v --compress_rasters=False

********************************
* Welcome to BreachDepressions *
* Powered by WhiteboxTools     *
* www.whiteboxgeo.com          *
********************************
Reading data...
Breaching in constrained mode...
Progress: 0%
Progress: 1%
Progress: 2%
Progress: 3%
Progress: 4%
Progress: 5%
Progress: 6%
Progress: 7%
Progress: 8%
Progress: 9%
Progress: 10%
Progress: 11%
Progress: 12%
Progress: 13%
Progress: 14%
Progress: 15%
Progress: 16%
Progress: 17%
Progress: 18%
Progress: 19%
Progress: 20%
Progress: 21%
Progress: 22%
Progress: 23%
Progress: 24%
Progress: 25%
Progress: 26%
Progress: 27%
Progress: 28%
Progress: 29%
Progress: 30%
Progres

In [ ]:
# Área da bacia (km²)
bacia_wm = bacia.to_crs(31982)  # métrico simples (ou use seu SRC projetado local)
bacia_wm["area_km2"] = bacia_wm.area / 1e6
print(bacia_wm[["area_km2"]])

In [ ]:
bacia_wm.plot()